# Ingesta y Procesamiento de Datos Agrícolas del SIAP (Sonora)

Esta libreta de Jupyter se utilizará para la ingesta, filtrado y limpieza de los datos históricos de producción agrícola, utilizando las bases de datos abiertas del Servicio de Información Agroalimentaria y Pesquera (SIAP) de la Secretaría de Agricultura y Desarrollo Rural (SADER).

## Ingesta de datos

### Librerías necesarias

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from loguru import logger
import pandas as pd
from pydantic import ValidationError
ROOT_DIR = Path.cwd().parent

if str(ROOT_DIR) not in sys.path:
  sys.path.append(str(ROOT_DIR))
  
from sonora_agriclimate_eda.schemas import SiapSonoraSchema
from sonora_agriclimate_eda.dataset import download_siap_sonora

start_year = 2019
end_year = 2023

# 1. Define paths from the notebook
root = Path().resolve().parent
INTERIM_DATA_DIR = root / "data/interim"
output_dir = INTERIM_DATA_DIR / "sonora_siap"

# 2. Ensure the folder exists and download directly.
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

## Descarga de datos

In [2]:
df_sonora = download_siap_sonora(start_year, end_year)

2026-09-29 10:53:29.745 | INFO     | sonora_agriclimate_eda.dataset:download_siap_sonora:291 - -> Conectando al portal del SIAP para obtener enlaces...
2026-09-29 10:53:30.482 | INFO     | sonora_agriclimate_eda.dataset:download_siap_sonora:303 - 
--- Procesando año: 2019 ---
2026-09-29 10:53:30.483 | INFO     | sonora_agriclimate_eda.dataset:download_siap_sonora:326 - -> Descargando CSV para 2019...
2026-09-29 10:53:32.442 | SUCCESS  | sonora_agriclimate_eda.dataset:download_siap_sonora:363 - ¡Éxito! Datos de Sonora (2019) guardados en: siap_sonora_cierre_2019.csv
2026-09-29 10:53:32.442 | INFO     | sonora_agriclimate_eda.dataset:download_siap_sonora:364 - Total de registros: 1245
2026-09-29 10:53:32.442 | INFO     | sonora_agriclimate_eda.dataset:download_siap_sonora:303 - 
--- Procesando año: 2020 ---
2026-09-29 10:53:32.443 | INFO     | sonora_agriclimate_eda.dataset:download_siap_sonora:326 - -> Descargando CSV para 2020...
2026-09-29 10:53:36.605 | SUCCESS  | sonora_agriclimate_

## Validación de la descarga de datos

### Función para validar los datos descargados

In [3]:
def validate_df(df: pd.DataFrame) -> pd.DataFrame:
    validated_rows = []

    for index, row in df.iterrows():
        try:
            agriculture = SiapSonoraSchema(**row.to_dict())
            validated_rows.append(agriculture.model_dump())

        except ValidationError as e:
            print(f"Error en fila {index}:")
            print(e)

    return pd.DataFrame(validated_rows)

In [4]:
# Get last generated file. Use max() to find the file with the latest modification time.
try:
  last_modified = max(
      (f for f in output_dir.iterdir() if f.is_file()),
      key=lambda f: f.stat().st_mtime,
  )
  print(f"Most recent file: {last_modified.name}")
  file_name = last_modified.name
except ValueError:
  print("The folder is empty or contains no files.")

Most recent file: siap_sonora_cierre_2023.csv


In [5]:
df_siap_sonora = pd.read_csv(str(output_dir)+"/" +file_name)
df_validated = validate_df(df_siap_sonora)
df_validated.head(5)

,Anio,Idestado,Idddr,Idmunicipio,Idciclo,Idmodalidad,Idcultivo,Nomestado,Nomddr,Nommunicipio,Nomciclo,Nommodalidad,Nomcultivo,Sembrada,Cosechada,Siniestrada,Produccion,Rendimiento,Pmr,Valorproduccion
0,2023,26,139,48,3,1,6650000,Sonora,Caborca,Puerto Peñasco,None,Riego,Espárrago,440.0,440.0,0.0,None,11.0,None,1.984274e+08
1,2023,26,139,70,1,1,5490000,Sonora,Caborca,General Plutarco Elías Calles,None,Riego,Avena forrajera en verde,10.0,10.0,0.0,None,18.0,None,1.447200e+05
2,2023,26,139,70,1,1,9050000,Sonora,Caborca,General Plutarco Elías Calles,None,Riego,Trigo grano,50.0,50.0,0.0,None,5.5,None,2.620062e+06
3,2023,26,139,70,1,1,15050000,Sonora,Caborca,General Plutarco Elías Calles,None,Riego,Pastos y praderas,60.0,60.0,0.0,None,18.0,None,9.720000e+05
4,2023,26,139,70,2,1,5190000,Sonora,Caborca,General Plutarco Elías Calles,None,Riego,Algodón hueso,20.0,20.0,0.0,None,4.2,None,1.325100e+06
